# 台語 AI 會議助理 — Colab 啟動

1. 執行階段選 **GPU**
2. 由上往下依序執行每個 cell

每次開新的 runtime 都要從頭跑一次（Colab 關掉後機器會被清空，資料庫內容也會消失）。

這份 notebook 全部用一般 Python 寫，不用 `%cd`、`!指令` 這類寫法，所以在瀏覽器版 Colab 和 VS Code 的 Colab 擴充功能都能跑。

## ⚡ 一鍵執行（測試時用這個就好）
一次跑完第 1、2、3、5 段（只讀雲端硬碟上名為 **test2** 的資料夾）：掛載雲端硬碟 → 複製最新程式 → 安裝（裝過就自動跳過）→ 啟動後端 → 開臨時網址。
最後會印出 👉 網址。要調整（例如不開臨時網址、強制重裝），改 `run_all.py` 最上面的選項。

執行完之後，第 4 段（看 log）、第 6 段（結束）照樣可以用。下面第 1～6 段保留，出問題時可以一段一段跑來找原因。

In [3]:
from google.colab import drive
import glob, os
drive.mount('/content/drive')
NAME = 'test2'   # 雲端硬碟上的專案資料夾名稱
pats = ['/content/drive/MyDrive/' + '*/' * d + NAME + '/run_all.py' for d in range(0, 3)]
pats += ['/content/drive/Othercomputers/' + '*/' * d + NAME + '/run_all.py' for d in range(1, 5)]
cands = sorted({p for pat in pats for p in glob.glob(pat)}, key=os.path.getmtime)
if not cands:
    raise FileNotFoundError('找不到 test2/run_all.py，請確認資料夾已改名為 test2 並同步完成')
exec(open(cands[-1], encoding='utf-8').read())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

① 掛載雲端硬碟、複製最新程式　（已執行 0 秒）
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 專案來源： /content/drive/MyDrive/test2
✅ 執行位置： /content/test2

② 安裝環境　（已執行 2 秒）
服務沒有在執行（資料庫 ✅、Ollama ❌）→ 重新執行安裝來啟動

==== [1/6] 安裝系統套件 (ffmpeg / PostgreSQL) ====
完成

==== [2/6] 安裝 Python 套件 ====
完成

==== [3/6] 啟動 PostgreSQL 並建立資料表 ====
 * Starting PostgreSQL 16 database server
   ...done.
psql:database/schema.sql:5: NOTICE:  relation "users" already exists, skipping
psql:database/schema.sql:12: NOTICE:  relation "sessions" already exists, skipping
psql:database/schema.sql:26: NOTICE:  relation "meetings" already exists, skipping
psql:database/schema.sql:36: NOTICE:  relation "meeting_transcripts" already exists, skipping
psql:database/schema.sql:53: NOTICE:  relation "meeting_summaries" already exists, skippi

In [ ]:
!cd /content/test2/backend && python ../scripts/seed_demo_meetings.py

## 1. 掛載 Google Drive，找到專案並複製到 Colab
會在「我的雲端硬碟」和「電腦」（Google 雲端硬碟電腦版同步的資料夾）裡尋找名為 **test2**、裡面有 `setup_colab.sh` 的專案資料夾（其他名稱的資料夾，例如 `all`、`all2`，不會被讀到），找到多份時用程式碼最新的那一份。

找到後會把程式**複製到 Colab 本機的 `/content/test2`** 再執行：同步的資料夾在 Colab 可能是唯讀，而且上傳的音檔、log 不會寫回你的電腦。第 3 段每次啟動後端前會再複製一次最新的程式。

In [ ]:
import glob, os, re, shutil, subprocess, sys, time, urllib.request
from google.colab import drive

drive.mount('/content/drive')

def sh(cmd, cwd=None):
    """執行 shell 指令並即時印出輸出（取代 ! 寫法，VS Code 也能用）。"""
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    return p.wait()

# ---- 1) 找專案：我的雲端硬碟（往下 2 層）＋ 電腦（Othercomputers，往下 4 層）
PROJECT_NAME = 'test2'   # 雲端硬碟上的專案資料夾名稱（只讀這個名稱的資料夾）
patterns = ['/content/drive/MyDrive/' + '*/' * d + PROJECT_NAME + '/setup_colab.sh' for d in range(0, 3)]
patterns += ['/content/drive/Othercomputers/' + '*/' * d + PROJECT_NAME + '/setup_colab.sh' for d in range(1, 5)]
found = sorted({os.path.dirname(p) for pat in patterns for p in glob.glob(pat)})
if not found:
    raise FileNotFoundError('在「我的雲端硬碟」和「電腦」裡都找不到 test2 資料夾（裡面要有 setup_colab.sh），請確認已改名並同步完成')

def newest_code_mtime(d):
    """專案裡程式碼最新的修改時間（用來判斷哪一份最新）。"""
    times = [os.path.getmtime(p) for sub in ('backend/app', 'frontend', 'scripts')
             for p in glob.glob(os.path.join(d, sub, '**', '*.*'), recursive=True)]
    return max(times, default=os.path.getmtime(os.path.join(d, 'setup_colab.sh')))

found.sort(key=newest_code_mtime)
SOURCE_DIR = found[-1]
if len(found) > 1:
    print('⚠️ 找到不只一份專案，使用程式碼最新的那一份（建議把其他舊的刪掉，避免混淆）：')
    for f in found:
        print('   ', '👉' if f == SOURCE_DIR else '  ', f, time.strftime('%m/%d %H:%M', time.localtime(newest_code_mtime(f))))

# ---- 2) 複製到 Colab 本機執行
PROJECT_DIR = '/content/' + PROJECT_NAME
BACKEND_DIR = os.path.join(PROJECT_DIR, 'backend')
LOG_DIR = os.path.join(PROJECT_DIR, 'logs')

def sync_code():
    """把雲端硬碟上的最新程式複製到 /content/test2（不動 log、上傳的音檔）。
    設定檔 .env 每次都依電腦上的版本重建：有 .env 用 .env，沒有就用 .env.example。
    所以要改設定（溫度、思考模式…），改電腦上的檔案，再重跑第 3 段就生效。"""
    shutil.copytree(SOURCE_DIR, PROJECT_DIR, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('logs', 'audio', '__pycache__', '*.pyc', '.env'))
    env_dst = os.path.join(PROJECT_DIR, '.env')
    for name in ('.env', '.env.example'):
        env_src = os.path.join(SOURCE_DIR, name)
        if os.path.exists(env_src):
            shutil.copy2(env_src, env_dst)
            break
    os.makedirs(LOG_DIR, exist_ok=True)
    os.makedirs(os.path.join(BACKEND_DIR, 'storage', 'audio'), exist_ok=True)

sync_code()
os.chdir(PROJECT_DIR)
print('✅ 專案來源：', SOURCE_DIR)
print('✅ 執行位置：', PROJECT_DIR)

## 2. 一鍵安裝全部套件、資料庫、LLM、ASR 模型
第一次約 10–20 分鐘（主要是下載 qwen3.8:27b）。最後會印出資料庫自我測試與環境檢查結果，全部 ✅ 才往下。

登入帳號來自 `.env` 的 `DEMO_USERNAME`、`DEMO_PASSWORD`。密碼至少 12 碼，而且要同時有英文字母和數字。`demo1234` 只有 8 碼，不能用來建立帳號。

In [ ]:
import os, re, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

SETUP = os.path.join(PROJECT_DIR, 'setup_colab.sh')

# 先把 Windows 換行（\r\n）轉成 Linux 換行，避免 bash 執行失敗
with open(SETUP, 'rb') as f:
    data = f.read()
with open(SETUP, 'wb') as f:
    f.write(data.replace(b'\r\n', b'\n'))

code = sh(['bash', SETUP])
print('\n✅ 安裝完成' if code == 0 else f'\n❌ 安裝失敗（代碼 {code}），請把上面最後幾十行貼出來')

## 3. 啟動後端
每次執行都會先複製雲端硬碟上的最新程式，所以程式改過、雲端同步完成後，**重跑這一段**就會套用（前端只改畫面時，瀏覽器重新整理也可以，但重跑這段最保險）。

正常 5～20 秒完成。啟動後會在背景載入 ASR 模型（約 1～2 分鐘），第一次上傳的辨識會稍微慢一點。

- **瀏覽器版 Colab**：cell 下方會嵌入網頁，另附開新視窗的連結。
- **VS Code**：下方不會出現網頁，請接著執行第 5 段取得網址。

In [ ]:
import json, os, re, shutil, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

# 先複製雲端硬碟上的最新程式（改過的檔案同步完成後，重跑這一段就會套用）
sync_code()
# 設定檔每次都依電腦上的版本重建（有 .env 用 .env，沒有就用 .env.example）
for name in ('.env', '.env.example'):
    if os.path.exists(os.path.join(SOURCE_DIR, name)):
        shutil.copy2(os.path.join(SOURCE_DIR, name), os.path.join(PROJECT_DIR, '.env'))
        break
print('已複製最新程式：', SOURCE_DIR, '→', PROJECT_DIR)

# 若之前已啟動過，先停掉舊的（app[.]main 的寫法是為了不要誤殺這個指令本身）
subprocess.run(['pkill', '-f', 'uvicorn app[.]main'])
time.sleep(1)

log = open(os.path.join(LOG_DIR, 'backend.log'), 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND_DIR, stdout=log, stderr=subprocess.STDOUT,
)

print('等待後端啟動', end='', flush=True)
ready = False
for _ in range(60):
    if server.poll() is not None:          # 程式已經結束 = 啟動時出錯
        break
    try:
        health = json.load(urllib.request.urlopen('http://127.0.0.1:8000/api/v1/health', timeout=2))
        ready = True
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(1)
print()

if ready:
    print('✅ 後端已啟動')
    print(f"   LLM：{health.get('llm_model')}　溫度 {health.get('llm_temperature')}　seed {health.get('llm_seed')}"
          f"　思考 {health.get('llm_think')}　上下文 {health.get('llm_num_ctx')}")
    print('   ※ 如果下面什麼都沒出現（例如用 VS Code），請改執行第 5 段取得網址')
    try:
        import warnings
        from google.colab import output
        output.serve_kernel_port_as_iframe(8000, height=900)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            output.serve_kernel_port_as_window(8000)
    except Exception as e:
        print('   （無法在這裡顯示網頁：', e, '）')
else:
    print('❌ 後端沒有啟動成功，以下是 log 最後 30 行：')
    with open(os.path.join(LOG_DIR, 'backend.log'), encoding='utf-8', errors='replace') as f:
        print(''.join(f.readlines()[-30:]))

## 4. 查看後端 log（出問題時用）

In [7]:
import os, re, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

with open(os.path.join(LOG_DIR, 'backend.log'), encoding='utf-8', errors='replace') as f:
    print(''.join(f.readlines()[-40:]))

INFO:     1.160.1.174:0 - "GET /app.js HTTP/1.1" 200 OK
INFO:     1.160.1.174:0 - "GET /img/favicon.svg HTTP/1.1" 200 OK
INFO:     1.160.1.174:0 - "POST /api/v1/auth/register HTTP/1.1" 200 OK
INFO:     1.160.1.174:0 - "GET /api/v1/health HTTP/1.1" 200 OK
INFO:     1.160.1.174:0 - "GET /img/oak-leaf-wide.jpg HTTP/1.1" 200 OK
INFO:     1.160.1.174:0 - "GET /api/v1/glossary HTTP/1.1" 200 OK
INFO:     1.160.1.174:0 - "GET /api/v1/glossary HTTP/1.1" 200 OK
2026-10-29 11:39:14,082 INFO app.services.meeting_service: 會議 259e41d6-3004-47cb-8175-d7fc05abe800 由上傳逐字稿建立，共 8 段
INFO:     1.160.1.174:0 - "POST /api/v1/meetings HTTP/1.1" 202 Accepted
INFO:     1.160.1.174:0 - "GET /api/v1/meetings/259e41d6-3004-47cb-8175-d7fc05abe800/status HTTP/1.1" 200 OK
INFO:     1.160.1.174:0 - "GET /api/v1/meetings/259e41d6-3004-47cb-8175-d7fc05abe800/proofread HTTP/1.1" 200 OK
INFO:     1.160.1.174:0 - "GET /api/v1/meetings/259e41d6-3004-47cb-8175-d7fc05abe800/transcript HTTP/1.1" 200 OK
INFO:     1.160.1.174:0 

## 5.（VS Code 使用者專用）用臨時網址開啟網頁

VS Code 的 Colab 擴充功能不會顯示第 3 段的網頁。這時執行下面這段，取得一個一般瀏覽器就能打開的網址。

⚠️ **隱私提醒**：網頁資料（音檔、逐字稿、重點）會經過 Cloudflare 轉送，且知道網址的人都能看到登入頁。這不符合專案「資料不經第三方」的原則，**只能用公開的測試音檔**，測完請執行第 6 段關閉。正式 Demo 請改用瀏覽器開 Colab。

In [2]:
import os, re, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

CLOUDFLARED = '/usr/local/bin/cloudflared'
if not os.path.exists(CLOUDFLARED):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)

subprocess.run(['pkill', '-f', 'cloudflared[ ]tunnel'])   # 關掉舊的
tunnel_log_path = '/content/tunnel.log'
tunnel = subprocess.Popen([CLOUDFLARED, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
                          stdout=open(tunnel_log_path, 'w'), stderr=subprocess.STDOUT)
# (a) 等 cloudflared 印出網址、並且跟 Cloudflare 連線成功
url = None
for _ in range(60):
    time.sleep(1)
    text = open(tunnel_log_path).read()
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', text)
    url = m.group(0) if m else None
    if url and 'Registered tunnel connection' in text:
        break
# (b) 從外部實際連一次，確定網址真的能用了才印出來（太早打開，電腦會記住「查不到」）
ok = False
if url:
    print('網址已建立，確認外部連得到', end='', flush=True)
    for _ in range(45):
        try:
            urllib.request.urlopen(url + '/api/v1/health', timeout=5)
            ok = True
            break
        except Exception:
            print('.', end='', flush=True)
            time.sleep(2)
    print()
if ok:
    print('👉 測試用網址：', url)
    print('   若顯示「無法連上這個網站」：等 30 秒再重新整理；仍不行，在 Windows 命令提示字元執行 ipconfig /flushdns')
    print('   每次重跑這段都會換新網址，舊網址會失效')
else:
    print('❌ 臨時網址沒有建立成功，log 如下：')
    print(open(tunnel_log_path).read()[-2000:])

網址已建立，確認外部連得到.....................................

KeyboardInterrupt: 

## 6. 結束
關閉後端與臨時網址。資料庫、Ollama 仍在背景執行，要重新開啟只要再跑第 3 段。
要完全釋放 GPU：「執行階段 → 中斷連線並刪除執行階段」（資料庫會清空，Drive 上的音檔保留）。

In [ ]:
import os, re, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

r1 = subprocess.run(['pkill', '-f', 'uvicorn app[.]main'])
print('後端已關閉' if r1.returncode == 0 else '後端沒有在執行')
r2 = subprocess.run(['pkill', '-f', 'cloudflared[ ]tunnel'])
print('臨時網址已關閉' if r2.returncode == 0 else '臨時網址沒有在執行')

## 附錄：查看資料庫內容

In [ ]:
import os, re, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

DB_CMD = ['psql', '-h', 'localhost', '-U', 'meeting', '-d', 'meeting_assistant', '-c']
env = dict(os.environ, PGPASSWORD='meeting')   # 若 .env 改過帳密，這裡也要改
for sql in [
    "SELECT meeting_name, source_type, status, created_at FROM meetings ORDER BY created_at DESC;",
    "SELECT m.meeting_name, t.transcript->>'segment_count' AS 段數 FROM meeting_transcripts t JOIN meetings m ON m.id = t.meeting_id;",
]:
    print(subprocess.run(DB_CMD + [sql], env=env, capture_output=True, text=True).stdout)

In [ ]:
import os, subprocess
env = dict(os.environ, PGPASSWORD='meeting')
sql = "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY 1;"
print(subprocess.run(
    ['psql', '-h', 'localhost', '-U', 'meeting', '-d', 'meeting_assistant', '-c', sql],
    env=env, capture_output=True, text=True,
).stdout)

In [ ]:
import os, subprocess
env = dict(os.environ, PGPASSWORD='meeting')
cmd = ['psql', '-h', 'localhost', '-U', 'meeting', '-d', 'meeting_assistant', '-c']
queries = [
    "SELECT username, id FROM users;",
    "SELECT meeting_name, source_type, status, date, start_time, end_time, created_at FROM meetings ORDER BY created_at DESC;",
    "SELECT m.meeting_name, t.transcript->>'segment_count' AS 段數, t.confirmed_at FROM meeting_transcripts t JOIN meetings m ON m.id = t.meeting_id;",
    "SELECT m.meeting_name, s.headline, s.language FROM meeting_summaries s JOIN meetings m ON m.id = s.meeting_id;",
    "SELECT term, created_at FROM user_terms ORDER BY created_at;",
    "SELECT username, failed_count, locked_until FROM login_locks;",
]
for sql in queries:
    print(subprocess.run(cmd + [sql], env=env, capture_output=True, text=True).stdout)